In [1]:
import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from analytics.forecasting.prophet import ProphetForecaster
from _pool_common import (
    load_pool_data,
    backtest_21d_rolling,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_PROPHET,
    ARTIFACTS_DIR,
    TEST_TICKERS,
)

In [2]:
stacked = load_pool_data()
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL    1256
JNJ     1256
JPM     1256
MSFT    1256
SPY     1256
WMT     1256
dtype: int64


,timestamp,symbol,close
0,2021-03-04,AAPL,120.129997
1,2021-03-05,AAPL,121.419998
2,2021-03-08,AAPL,116.360001
3,2021-03-09,AAPL,121.089996
4,2021-03-10,AAPL,119.980003
5,2021-03-11,AAPL,121.959999
6,2021-03-12,AAPL,121.029999
7,2021-03-15,AAPL,123.989998
8,2021-03-16,AAPL,125.570000
9,2021-03-17,AAPL,124.760002


In [3]:
def get_forecast_prophet(context: pd.Series, horizon: int):
    model = ProphetForecaster(confidence_level=0.95)
    model.fit(context)
    fc = model.forecast(periods=horizon)
    point = (fc.get("point_forecast") or []) if fc else []
    return [float(x) for x in point] if point else []

model_name = "prophet"
all_preds = []
for sym in TEST_TICKERS:
    grp = stacked[stacked["symbol"] == sym]
    if grp.empty:
        continue
    prices = grp.set_index("timestamp")["close"].astype(float).dropna().sort_index()
    if len(prices) < TEST_SIZE + MIN_TRAIN_PROPHET:
        continue
    pred = backtest_21d_rolling(
        prices, TEST_SIZE, FORECAST_HORIZON, ROLLING_STEP,
        MIN_TRAIN_PROPHET,
        get_forecast_fn=get_forecast_prophet,
    )
    if pred.empty:
        continue
    pred["symbol"] = sym
    all_preds.append(pred)

pred_prophet = pd.concat(all_preds, ignore_index=True)
print(pred_prophet.groupby("symbol").size())
pred_prophet.head()

c:\capstone_project_unfc\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.
15:56:33 - cmdstanpy - INFO - Chain [1] start processing
15:56:35 - cmdstanpy - INFO - Chain [1] done processing
15:56:36 - cmdstanpy - INFO - Chain [1] start processing
15:56:36 - cmdstanpy - INFO - Chain [1] done processing
15:56:37 - cmdstanpy - INFO - Chain [1] start processing
15:56:37 - cmdstanpy - INFO - Chain [1] done processing
15:56:38 - cmdstanpy - INFO - Chain [1] start processing
15:56:38 - cmdstanpy - INFO - Chain [1] done processing
15:56:39 - cmdstanpy - INFO - Chain [1] start processing
15:56:39 - cmdstanpy - INFO - Chain [1] done processing
15:56:40 - cmdstanpy - INFO - Chain [1] start processing
15:56:40 - cmdstanpy - INFO - Chain [1] done processing
15:56:41 - c

symbol
MSFT    126
SPY     126
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-05,483.160004,527.0835,0,MSFT
1,2025-12-08,491.019989,507.6594,0,MSFT
2,2025-12-09,492.019989,508.5362,0,MSFT
3,2025-12-10,478.559998,529.5715,0,MSFT
4,2025-12-11,483.470001,529.9201,0,MSFT


In [4]:
metrics_rows = []
for sym in pred_prophet["symbol"].unique():
    sub = pred_prophet[pred_prophet["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_prophet)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_prophet_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_prophet_pool.parquet")

     model   symbol        MAE       RMSE     MAPE_%
0  prophet     MSFT  57.415211  61.097248  13.189283
1  prophet      SPY  11.616599  15.352457   1.687281
2  prophet  overall  34.515905  44.771194   7.438282
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_prophet_pool.parquet
